(centiles-calculation)=
# Centiles Calculation

### gamlss_main.Gamlss.<br>**centiles**

```python
centiles(test_data, train_data, centiles=[3, 10, 25, 50, 75, 90, 97], func_name=None)
```

&nbsp;&nbsp;&nbsp;&nbsp;Calculates specified centiles for each observation based on the fitted model parameters. Output shape is (number of samples, number of centiles).

&nbsp;&nbsp;&nbsp;&nbsp;<span style="background-color: #f5f5f5; padding: 4px 8px; display: inline-block; width: 90%;">**Parameters:**</span>

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**test_data** : ***pandas.DataFrame*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;DataFrame containing the test data

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**train_data** : ***pandas.DataFrame*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;DataFrame containing the training data

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**centiles** : ***list***, *default=[3, 10, 25, 50, 75, 90, 97]* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;List of centiles to calculate, in percent: between 0 and 100, not 0 or 100 themselves (e.g. 2.5 or 97.5). If all values are below 1 (e.g. 0.03), a warning reminds you that the 3rd centile is 3

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**func_name** : ***str***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Distribution family name. If None (default), the model's own family is used; a different family raises an error

&nbsp;&nbsp;&nbsp;&nbsp;<span style="background-color: #f5f5f5; padding: 4px 8px; display: inline-block; width: 90%;">**Returns:**</span>

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**centile_values** : ***numpy.ndarray*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Array of shape (n_samples, n_centiles) with centile values

<br>


**Examples:**

In [31]:
# Centile calculation
centile_score = gamlss_model.centiles(df_test,df_train,func_name = 'SHASH', centiles = [3, 10, 25, 50, 75, 90, 97])
# Output is (n_samples, n_centiles)
print(f"Shape of centile scores: {centile_score.shape}")
print(centile_score)

Shape of centile scores: (470, 7)
[[22.26361183 22.70404629 23.29841832 ... 25.03367042 25.63883369
  26.08781171]
 [22.18474533 22.62521059 23.21962418 ... 24.95499763 25.56020321
  26.00921264]
 [20.76278981 21.20411026 21.79967795 ... 23.53842074 24.14480137
  24.59468269]
 ...
 [21.3364953  21.7773827  22.37236599 ... 24.10940262 24.71518824
  25.16462806]
 [21.15285185 21.59437228 22.19020985 ... 23.92974055 24.53639596
  24.98648117]
 [21.4493725  21.89059435 22.48602897 ... 24.22438328 24.83062844
  25.28040923]]


### gamlss_main.Gamlss.<br>**create_site_adjusted_data**

```python
create_site_adjusted_data(train_data, reference_site, correction_col_name='site',
                          model_train_data=None)
```

&nbsp;&nbsp;&nbsp;&nbsp;Performs robust site correction by extracting the site random effects (`re()` terms) of every parameter of the fitted GAMLSS model and adjusting data to a reference site. Creates a new column '{y_val}_adjusted' with site-corrected values while preserving all original data.

&nbsp;&nbsp;&nbsp;&nbsp;How the values are moved depends on the model:

- **Site only in mu, identity link, location family** (e.g. NO, SHASH, TF): each value is shifted by the difference between its site's effect and the reference site's effect. This is a plain shift.
- **Otherwise** (site also in sigma, nu or tau, a log link, or a family where mu is not a location, e.g. GA): each value keeps its centile. A value on the 80th centile of its own site's distribution moves to the 80th centile of the reference site's distribution.

&nbsp;&nbsp;&nbsp;&nbsp;If the model has no `re()` site term, a warning is given and nothing is adjusted.

&nbsp;&nbsp;&nbsp;&nbsp;<span style="background-color: #f5f5f5; padding: 4px 8px; display: inline-block; width: 90%;">**Parameters:**</span>

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**train_data** : ***pandas.DataFrame*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Original training data containing the site/group information

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**reference_site** : ***str*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Site to use as reference (all other sites adjusted to this baseline). It must be one of the model's sites; any other value raises an error

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**correction_col_name** : ***str***, *default='site'* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Name of the column containing site/group information for correction

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**model_train_data** : ***pandas.DataFrame***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;The data the model was fitted on. Only needed when `train_data` is other data than the model's own training data and the adjustment is not a plain shift (the centile mapping needs the model's predictions for those rows). `plot_centiles(site_correction=True)` passes it for you

&nbsp;&nbsp;&nbsp;&nbsp;<span style="background-color: #f5f5f5; padding: 4px 8px; display: inline-block; width: 90%;">**Returns:**</span>

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**adjusted_data** : ***pandas.DataFrame*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Same as input with additional '{y_val}_adjusted' column containing site-corrected values

<br>

**Examples:**


In [32]:
# Site correction with custom column name
adjusted_data = gamlss_model.create_site_adjusted_data(
    train_data=train_data1, 
    reference_site='HJDXM',
    correction_col_name='site'
)

# The adjusted data will have both original and corrected y values
print(f"Original columns: {df_train.columns.tolist()}")
print(f"Adjusted columns: {adjusted_data.columns.tolist()}")
print(f"New column: {gamlss_model.y_val}_adjusted")

=== Robust Random Effects Extraction for 'site' ===

Checking mu parameter...
Found random effects in mu parameter:
         offset  group
BCCLF -0.004787  BCCLF
ESZYC -0.002687  ESZYC
HJDXM -0.005445  HJDXM
HLMEG  0.003174  HLMEG
HSREL -0.004600  HSREL
KEMUB  0.001695  KEMUB
TIXLZ  0.014594  TIXLZ
ZSCPY -0.001944  ZSCPY

Checking sigma parameter...
Found random effects in sigma parameter:
             offset  group
BCCLF  1.750685e-08  BCCLF
ESZYC  8.567566e-08  ESZYC
HJDXM -8.668942e-08  HJDXM
HLMEG  1.065463e-08  HLMEG
HSREL -5.122056e-08  HSREL
KEMUB -5.814322e-09  KEMUB
TIXLZ  3.700854e-08  TIXLZ
ZSCPY -7.121374e-09  ZSCPY

Checking nu parameter...
No random effects in nu

Checking tau parameter...
No random effects in tau

 Using random effects from mu, sigma parameter(s)

 Using 'HJDXM' as reference site (mu offset = -0.005445)

 Using 'HJDXM' as reference site (sigma offset = -0.000000)

 Applying site corrections:
  HJDXM: mu offset = 0.000000, sigma offset = 0.000000, n = 210